# 🏭 Smart Industrial Maintenance System — Full Pipeline
## FSE 570 Capstone | Arizona State University

**End-to-end predictive maintenance** using the **NASA IMS Bearing Dataset** (~2GB).

### Models:
1. **LSTM Temporal Autoencoder** — Anomaly Detection
2. **LSTM Predictor with Attention** — Failure Probability
3. **XGBoost Regression** — Remaining Useful Life (RUL)
4. **Bayesian Weibull Survival** — Time-to-Failure with Uncertainty
5. **MILP Optimizer** — Maintenance Scheduling

### Pipeline:
`Raw Vibration → Feature Extraction → Model Training → Inference → MILP Scheduling → Visualization`

> ⚠️ **Runtime**: Set to **T4 GPU** via `Runtime → Change runtime type → T4 GPU`

---
## 1. Environment Setup

In [1]:
# Install dependencies
!pip install -q torch torchvision xgboost lifelines shap pulp scipy scikit-learn matplotlib seaborn kagglehub jupytext


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\devas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats as sp_stats
from scipy.fft import rfft, rfftfreq
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             f1_score, precision_score, recall_score, roc_auc_score)
import xgboost as xgb
from lifelines import WeibullAFTFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from pulp import (LpProblem, LpMinimize, LpVariable, LpBinary,
                  lpSum, PULP_CBC_CMD, LpStatus, value)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🔧 Device: cpu


---
## 2. Configuration

In [3]:
class Config:
    """Central configuration for the entire pipeline."""
    # Paths
    PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))
    # When running from notebooks/, go up one level to the repo root
    if os.path.basename(PROJECT_ROOT) == "notebooks":
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
    DATA_DIR = os.path.join(PROJECT_ROOT, "data")
    RAW_IMS_DIR = os.path.join(DATA_DIR, "raw_ims")
    PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
    MODELS_DIR = os.path.join(PROJECT_ROOT, os.path.join("models", "saved"))

    for d in [RAW_IMS_DIR, PROCESSED_DIR, MODELS_DIR]:
        os.makedirs(d, exist_ok=True)

    # Device
    DEVICE = DEVICE
    RANDOM_SEED = 42

    # IMS Dataset
    IMS_DATASET = "vinayak123tyagi/bearing-dataset"
    IMS_SAMPLING_RATE = 20480
    IMS_SNAPSHOT_LENGTH = 20480
    IMS_EXPERIMENTS = {
        1: {"channels": 8, "bearings": 4, "failed_bearings": [3, 4],
            "failure_modes": ["inner_race", "roller_element"], "folder": "1st_test"},
        2: {"channels": 4, "bearings": 4, "failed_bearings": [1],
            "failure_modes": ["outer_race"], "folder": "2nd_test"},
        3: {"channels": 4, "bearings": 4, "failed_bearings": [3],
            "failure_modes": ["outer_race"], "folder": "3rd_test"},
    }

    # Preprocessing
    IMS_MAX_RUL = 125
    IMS_SEQUENCE_LENGTH = 30
    IMS_FFT_BANDS = 5
    IMS_ROLLING_WINDOWS = [10, 50, 100]
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    MAX_RUL = 125

    # LSTM Autoencoder
    AE_HIDDEN_DIM = 64
    AE_LATENT_DIM = 32
    AE_NUM_LAYERS = 2
    AE_DROPOUT = 0.2
    AE_LEARNING_RATE = 1e-3
    AE_EPOCHS = 50
    AE_BATCH_SIZE = 64
    AE_ANOMALY_THRESHOLD_SIGMA = 3.0

    # LSTM Predictor
    PRED_HIDDEN_DIM = 64
    PRED_NUM_LAYERS = 2
    PRED_DROPOUT = 0.3
    PRED_LEARNING_RATE = 1e-3
    PRED_EPOCHS = 50
    PRED_BATCH_SIZE = 64
    PRED_FAILURE_HORIZON = 30

    # XGBoost
    XGB_PARAMS = {
        "n_estimators": 200, "max_depth": 6, "learning_rate": 0.1,
        "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3,
        "reg_alpha": 0.1, "reg_lambda": 1.0, "random_state": 42,
    }
    if torch.cuda.is_available():
        XGB_PARAMS["device"] = "cuda"
        XGB_PARAMS["tree_method"] = "hist"

    # Survival
    SURVIVAL_CONFIDENCE_LEVELS = [0.90, 0.95]

    # MILP
    MAX_CONCURRENT_CREWS = 3
    DOWNTIME_COST_PER_HOUR = 10000
    MAINTENANCE_COST_BASE = 2000
    SAFETY_RISK_THRESHOLD = 0.7
    SCHEDULING_HORIZON = 10

    # Risk
    RISK_LEVELS = {
        "critical": {"threshold": 0.7, "label": "Service Immediately", "color": "#FF4444"},
        "elevated": {"threshold": 0.4, "label": "Schedule Soon", "color": "#FFAA00"},
        "normal":   {"threshold": 0.0, "label": "Continue Monitoring", "color": "#44BB44"},
    }

cfg = Config()
print("✅ Configuration loaded")

✅ Configuration loaded


---
## 3. Download IMS Bearing Dataset (~2 GB)

Uses `kagglehub` for automatic download — no manual API key upload needed.
On first use, kagglehub will prompt for your Kaggle credentials and cache them.

> **Note**: `kagglehub` handles authentication automatically.
> It reads `~/.kaggle/kaggle.json` if present, or prompts you for credentials on first use.

In [4]:
def _find_data_dir(base_path, folder_hint):
    """Find the deepest directory containing actual data files."""
    best_dir, best_count = None, 0
    for root, dirs, files in os.walk(base_path):
        data_files = [f for f in files
                      if not f.startswith('.') and not f.endswith('.zip')
                      and not f.endswith('.pdf')]
        if data_files and len(data_files) > best_count:
            rel = root.replace(base_path, '').strip(os.sep)
            parts = rel.split(os.sep)
            if parts and parts[0] == folder_hint:
                best_dir, best_count = root, len(data_files)
    return best_dir, best_count

def download_ims_dataset():
    """Download the full IMS bearing dataset via kagglehub (~2GB)."""
    existing_valid = []
    for d in os.listdir(cfg.RAW_IMS_DIR):
        dp = os.path.join(cfg.RAW_IMS_DIR, d)
        if os.path.isdir(dp):
            n = len([f for f in os.listdir(dp) if os.path.isfile(os.path.join(dp, f))])
            if n > 0:
                existing_valid.append((d, n))
    if len(existing_valid) >= 3:
        print(f"\u2705 Dataset already present ({len(existing_valid)} folders)")
        for d, n in sorted(existing_valid):
            print(f"   \U0001f4c2 {d}/ ({n} files)")
        return

    try:
        import kagglehub
        print("\U0001f4e5 Downloading IMS bearing dataset via kagglehub...")
        path = kagglehub.dataset_download("vinayak123tyagi/bearing-dataset")
        print(f"   Downloaded to: {path}")
        import shutil
        for exp_id, info in cfg.IMS_EXPERIMENTS.items():
            folder = info["folder"]
            exp_path = os.path.join(cfg.RAW_IMS_DIR, folder)
            os.makedirs(exp_path, exist_ok=True)
            data_dir, file_count = _find_data_dir(path, folder)
            if data_dir and file_count > 0:
                print(f"   \U0001f4c2 Found {file_count} files for experiment {exp_id}")
                for f in os.listdir(data_dir):
                    src = os.path.join(data_dir, f)
                    dst = os.path.join(exp_path, f)
                    if os.path.isfile(src) and not os.path.exists(dst):
                        try:
                            os.symlink(src, dst)
                        except (OSError, NotImplementedError):
                            shutil.copy2(src, dst)
                n = len([f for f in os.listdir(exp_path) if os.path.isfile(os.path.join(exp_path, f))])
                print(f"   \u2705 Experiment {exp_id}: {n} files in {folder}/")
            else:
                print(f"   \u26a0\ufe0f Experiment {exp_id}: no data files found")
    except Exception as e:
        print(f"\u274c kagglehub download failed: {e}")
        print("\U0001f504 Generating synthetic IMS data...")
        _generate_synthetic_ims()

download_ims_dataset()

📥 Downloading IMS bearing dataset via kagglehub...
   Downloaded to: C:\Users\devas\.cache\kagglehub\datasets\vinayak123tyagi\bearing-dataset\versions\1
   📂 Found 2156 files for experiment 1
   ✅ Experiment 1: 2156 files in 1st_test/
   📂 Found 984 files for experiment 2
   ✅ Experiment 2: 984 files in 2nd_test/
   📂 Found 6324 files for experiment 3
   ✅ Experiment 3: 6324 files in 3rd_test/


---
## 4. Load & Parse IMS Data

In [5]:
def load_ims_experiment(experiment=2):
    """Load raw vibration snapshots from one IMS experiment."""
    info = cfg.IMS_EXPERIMENTS[experiment]
    exp_path = os.path.join(cfg.RAW_IMS_DIR, info["folder"])
    if not os.path.isdir(exp_path):
        raise FileNotFoundError(f"Experiment {experiment} not found at {exp_path}")

    files = sorted([f for f in os.listdir(exp_path)
                    if os.path.isfile(os.path.join(exp_path, f)) and not f.startswith('.')])
    n_ch = info["channels"]
    n_b = info["bearings"]

    if n_ch == 8:
        ch_names = []
        for b in range(1, n_b + 1):
            ch_names.extend([f"bearing{b}_x", f"bearing{b}_y"])
    else:
        ch_names = [f"bearing{b}" for b in range(1, n_b + 1)]

    print(f"📂 Loading experiment {experiment}: {len(files)} files, {n_ch} channels")

    snapshots = []
    for fi, fn in enumerate(files):
        fp = os.path.join(exp_path, fn)
        try:
            data = np.loadtxt(fp, delimiter='\t')
            if data.ndim == 1:
                data = data.reshape(-1, 1)
            snap = {"file_index": fi, "filename": fn}
            actual_ch = min(data.shape[1], n_ch)
            for ci in range(actual_ch):
                snap[ch_names[ci]] = data[:, ci]
            snapshots.append(snap)
        except Exception as e:
            if fi == 0:
                print(f"   ⚠️ Could not load {fn}: {e}")

    print(f"   ✅ Loaded {len(snapshots)} snapshots")
    return snapshots, ch_names, info

# Load Experiment 2 (clearest single-bearing outer race failure)
snapshots, channel_names, exp_info = load_ims_experiment(experiment=2)

📂 Loading experiment 2: 984 files, 4 channels
   ✅ Loaded 984 snapshots


---
## 5. Vibration Feature Extraction

Extract **19 features per channel** from each 1-second vibration snapshot:
- **Time-domain**: RMS, peak, peak-to-peak, crest factor, kurtosis, skewness, std
- **Frequency-domain**: dominant frequency, spectral centroid, 5 energy bands, spectral kurtosis

In [6]:
class IMSFeatureExtractor:
    """Extracts health features from raw vibration signals."""

    def __init__(self, sampling_rate=None, n_fft_bands=None):
        self.sr = sampling_rate or cfg.IMS_SAMPLING_RATE
        self.n_bands = n_fft_bands or cfg.IMS_FFT_BANDS

    def extract_time_features(self, signal):
        rms = np.sqrt(np.mean(signal**2))
        peak = np.max(np.abs(signal))
        return {
            "rms": rms, "peak": peak,
            "peak_to_peak": np.max(signal) - np.min(signal),
            "crest_factor": peak / (rms + 1e-10),
            "kurtosis": sp_stats.kurtosis(signal, fisher=True),
            "skewness": sp_stats.skew(signal),
            "std": np.std(signal),
        }

    def extract_freq_features(self, signal):
        n = len(signal)
        fft_vals = np.abs(rfft(signal))[1:]
        fft_freqs = rfftfreq(n, d=1.0/self.sr)[1:]
        total_E = np.sum(fft_vals**2)
        dom_idx = np.argmax(fft_vals)

        result = {
            "dominant_freq": fft_freqs[dom_idx],
            "spectral_centroid": np.sum(fft_freqs * fft_vals) / (np.sum(fft_vals) + 1e-10),
            "spectral_kurtosis": sp_stats.kurtosis(fft_vals, fisher=True),
        }
        edges = np.linspace(0, fft_freqs[-1], self.n_bands+1)
        for i in range(self.n_bands):
            mask = (fft_freqs >= edges[i]) & (fft_freqs < edges[i+1])
            result[f"band_{i+1}_energy"] = np.sum(fft_vals[mask]**2) / (total_E + 1e-10)
        return result

    def extract_snapshot_features(self, snap, ch_names):
        feats = {}
        for ch in ch_names:
            if ch not in snap or not isinstance(snap[ch], np.ndarray) or len(snap[ch]) < 10:
                continue
            sig = snap[ch]
            for k, v in self.extract_time_features(sig).items():
                feats[f"{ch}_{k}"] = v
            for k, v in self.extract_freq_features(sig).items():
                feats[f"{ch}_{k}"] = v
        return feats

In [7]:
# Extract features from all snapshots
extractor = IMSFeatureExtractor()
print(f"⚙️ Extracting features from {len(snapshots)} snapshots...")
t0 = time.time()

rows = []
for snap in snapshots:
    feats = extractor.extract_snapshot_features(snap, channel_names)
    feats["file_index"] = snap["file_index"]
    rows.append(feats)

df_features = pd.DataFrame(rows).sort_values("file_index").reset_index(drop=True)

# Assign pseudo-RUL
total = len(df_features)
df_features["RUL"] = total - df_features["file_index"] - 1
df_features["RUL"] = df_features["RUL"].clip(upper=cfg.IMS_MAX_RUL)
df_features["unit_id"] = 1

elapsed = time.time() - t0
n_feat = len([c for c in df_features.columns if c not in ["file_index", "RUL", "unit_id"]])
print(f"✅ Extracted {n_feat} features per snapshot in {elapsed:.1f}s")
print(f"   RUL range: [{df_features['RUL'].min()}, {df_features['RUL'].max()}]")
print(f"   Shape: {df_features.shape}")

⚙️ Extracting features from 984 snapshots...
✅ Extracted 60 features per snapshot in 7.0s
   RUL range: [0, 500]
   Shape: (984, 63)


---
## 6. Add Rolling Trend Features & Preprocess

In [8]:
# Add rolling RMS trend features
rms_cols = [c for c in df_features.columns if c.endswith("_rms")]
for col in rms_cols:
    for w in cfg.IMS_ROLLING_WINDOWS:
        df_features[f"{col}_roll{w}_mean"] = df_features[col].rolling(w, min_periods=1).mean()
        df_features[f"{col}_roll{w}_std"] = df_features[col].rolling(w, min_periods=1).std().fillna(0)

print(f"✅ Added {len(rms_cols) * len(cfg.IMS_ROLLING_WINDOWS) * 2} rolling features")

# Drop low-variance features
exclude = ["file_index", "unit_id", "RUL"]
feat_cols = [c for c in df_features.columns if c not in exclude]
low_var = [c for c in feat_cols if df_features[c].std() < 0.001]
if low_var:
    df_features = df_features.drop(columns=low_var)
    print(f"🗑️ Dropped {len(low_var)} low-variance features: {low_var[:5]}...")

df_features = df_features.fillna(0)
print(f"📊 Final feature shape: {df_features.shape}")

✅ Added 24 rolling features
📊 Final feature shape: (984, 87)


In [9]:
# Data splitting: Use interleaved split to ensure all phases of degradation
# are represented in train, val, and test sets.
# For a single time-series, we sample subsequences across the full timeline.

n = len(df_features)

# Create interleaved indices: every 5th sample for test, every 5th of remainder for val
indices = np.arange(n)
test_idx = indices[::7]  # ~14% for test
remaining = np.array([i for i in indices if i not in test_idx])
val_idx = remaining[::6]  # ~14% for val
train_idx = np.array([i for i in remaining if i not in val_idx])  # ~72% for train

df_train = df_features.iloc[train_idx].copy()
df_val = df_features.iloc[val_idx].copy()
df_test = df_features.iloc[test_idx].copy()
print(f"\u2705 Split: train={len(df_train)}, val={len(df_val)}, test={len(df_test)}")
print(f"   Train RUL range: [{df_train['RUL'].min()}, {df_train['RUL'].max()}]")
print(f"   Val RUL range:   [{df_val['RUL'].min()}, {df_val['RUL'].max()}]")
print(f"   Test RUL range:  [{df_test['RUL'].min()}, {df_test['RUL'].max()}]")

# Normalize
scaler = MinMaxScaler()
feat_cols = [c for c in df_features.columns if c not in exclude]
df_train[feat_cols] = scaler.fit_transform(df_train[feat_cols])
df_val[feat_cols] = scaler.transform(df_val[feat_cols])
df_test[feat_cols] = scaler.transform(df_test[feat_cols])

# For LSTM: create contiguous sequences from the FULL timeline, then split
# This preserves temporal context while mixing healthy/degraded states
all_X_data = scaler.transform(df_features[feat_cols].values)
all_rul = df_features["RUL"].values

all_seqs, all_labs, all_indices = [], [], []
seq_len = cfg.IMS_SEQUENCE_LENGTH
for i in range(len(all_X_data) - seq_len + 1):
    all_seqs.append(all_X_data[i:i+seq_len])
    all_labs.append(all_rul[i+seq_len-1])
    all_indices.append(i + seq_len - 1)  # index of the label

all_seqs = np.array(all_seqs, dtype=np.float32)
all_labs = np.array(all_labs, dtype=np.float32)
all_indices = np.array(all_indices)

# Split sequences by their label index
train_mask = np.isin(all_indices, train_idx)
val_mask = np.isin(all_indices, val_idx)
test_mask = np.isin(all_indices, test_idx)

X_train, y_train_rul = all_seqs[train_mask], all_labs[train_mask]
X_val, y_val_rul = all_seqs[val_mask], all_labs[val_mask]
X_test, y_test_rul = all_seqs[test_mask], all_labs[test_mask]

y_train_bin = (y_train_rul <= cfg.PRED_FAILURE_HORIZON).astype(np.float32)
y_val_bin = (y_val_rul <= cfg.PRED_FAILURE_HORIZON).astype(np.float32)
y_test_bin = (y_test_rul <= cfg.PRED_FAILURE_HORIZON).astype(np.float32)

n_features = X_train.shape[2]
print(f"\n\u2705 Sequences ready:")
for name, X, y in [("Train", X_train, y_train_rul), ("Val", X_val, y_val_rul), ("Test", X_test, y_test_rul)]:
    bin_rate = (y <= cfg.PRED_FAILURE_HORIZON).mean()
    print(f"   {name}: X={X.shape}, RUL=[{y.min():.0f}, {y.max():.0f}], failure_rate={bin_rate:.1%}")


📊 Split: train=688, val=147, test=149

✅ Sequences ready:
   Train: X=(639, 50, 84), RUL=[296, 500]
   Val: X=(98, 50, 84), RUL=[149, 246]
   Test: X=(100, 50, 84), RUL=[0, 99]


---
## 7. Model 1: LSTM Temporal Autoencoder (Anomaly Detection)

In [10]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

class LSTMDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, seq_len, num_layers, dropout):
        super().__init__()
        self.seq_len = seq_len
        self.fc = nn.Linear(latent_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, latent):
        h = self.fc(latent).unsqueeze(1).repeat(1, self.seq_len, 1)
        o, _ = self.lstm(h)
        return self.out(o)

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden=64, latent=32, layers=2, dropout=0.2, seq_len=50):
        super().__init__()
        self.encoder = LSTMEncoder(input_dim, hidden, latent, layers, dropout)
        self.decoder = LSTMDecoder(latent, hidden, input_dim, seq_len, layers, dropout)
        self.threshold = None
        self.input_dim = input_dim

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def compute_anomaly_score(self, x):
        self.eval()
        with torch.no_grad():
            x = x.to(cfg.DEVICE)
            recon = self.forward(x)
            return ((x - recon)**2).mean(dim=(1,2)).cpu().numpy()

    def set_threshold(self, scores, sigma=3.0):
        self.threshold = np.mean(scores) + sigma * np.std(scores)
        print(f"   Threshold: {self.threshold:.6f}")
        return self.threshold

    def detect_anomalies(self, x):
        scores = self.compute_anomaly_score(x)
        return scores, scores > self.threshold

In [11]:
# Train Autoencoder on healthy data
print("=" * 60)
print("\U0001f534 TRAINING: LSTM AUTOENCODER (Anomaly Detection)")
print("=" * 60)

ae = LSTMAutoencoder(n_features, seq_len=cfg.IMS_SEQUENCE_LENGTH).to(cfg.DEVICE)
ae_opt = torch.optim.Adam(ae.parameters(), lr=cfg.AE_LEARNING_RATE)
ae_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(ae_opt, patience=5, factor=0.5)
ae_criterion = nn.MSELoss()

# Use high-RUL data as "healthy" (top 60% of lifespan)
healthy_mask = y_train_rul > cfg.IMS_MAX_RUL * 0.6
X_healthy = X_train[healthy_mask]
print(f"   Healthy samples: {len(X_healthy)}/{len(X_train)} ({len(X_healthy)/len(X_train):.0%})")

if len(X_healthy) < 10:
    # Fallback: use top 50%
    healthy_mask = y_train_rul > np.median(y_train_rul)
    X_healthy = X_train[healthy_mask]
    print(f"   Fallback healthy: {len(X_healthy)}/{len(X_train)}")

loader = DataLoader(TensorDataset(torch.FloatTensor(X_healthy), torch.FloatTensor(X_healthy)),
                    batch_size=cfg.AE_BATCH_SIZE, shuffle=True)
for epoch in range(cfg.AE_EPOCHS):
    ae.train()
    total_loss = 0
    for bx, _ in loader:
        bx = bx.to(cfg.DEVICE)
        recon = ae(bx)
        loss = ae_criterion(recon, bx)
        ae_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
        ae_opt.step()
        total_loss += loss.item() * len(bx)
    avg = total_loss / len(X_healthy)
    ae_sched.step(avg)
    if (epoch+1) % 10 == 0:
        print(f"   Epoch {epoch+1}/{cfg.AE_EPOCHS} \u2014 Loss: {avg:.6f}")

train_scores = ae.compute_anomaly_score(torch.FloatTensor(X_healthy))
ae.set_threshold(train_scores)
torch.save({"model_state": ae.state_dict(), "threshold": ae.threshold,
            "input_dim": n_features}, os.path.join(cfg.MODELS_DIR, "ims_autoencoder.pt"))
print("\u2705 Autoencoder trained and saved")


🔴 TRAINING: LSTM AUTOENCODER (Anomaly Detection)
   Healthy samples: 639/639
   Epoch 10/50 — Loss: 0.026762
   Epoch 20/50 — Loss: 0.026199
   Epoch 30/50 — Loss: 0.026009
   Epoch 40/50 — Loss: 0.025932
   Epoch 50/50 — Loss: 0.020465
   Threshold: 0.034638
✅ Autoencoder trained and saved


---
## 8. Model 2: LSTM Failure Predictor with Attention

In [12]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_dim, hidden=64, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0)
        self.attention = nn.Sequential(nn.Linear(hidden, hidden//2), nn.Tanh(),
                                       nn.Linear(hidden//2, 1))
        self.classifier = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(hidden//2, 1))
        self.hidden_dim = hidden

    def forward(self, x):
        out, _ = self.lstm(x)
        attn = torch.softmax(self.attention(out), dim=1)
        context = torch.sum(out * attn, dim=1)
        logits = self.classifier(context)
        return logits.squeeze(-1), attn.squeeze(-1)

    def predict_proba(self, x):
        self.eval()
        with torch.no_grad():
            x = x.to(cfg.DEVICE)
            logits, attn = self.forward(x)
            return torch.sigmoid(logits).cpu().numpy(), attn.cpu().numpy()

In [13]:
print("=" * 60)
print("\U0001f3c3 TRAINING: LSTM FAILURE PREDICTOR (With Attention)")
print("=" * 60)

pred = LSTMPredictor(n_features).to(cfg.DEVICE)
pred_opt = torch.optim.Adam(pred.parameters(), lr=cfg.PRED_LEARNING_RATE)

pos_count = y_train_bin.sum()
neg_count = len(y_train_bin) - pos_count
pos_weight = torch.tensor([neg_count / max(pos_count, 1)]).to(cfg.DEVICE)
pred_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"   Samples: {len(y_train_bin)}, positive: {int(pos_count)} ({pos_count/len(y_train_bin):.1%}), weight: {pos_weight.item():.2f}")

if pos_count == 0:
    print("   \u26a0\ufe0f No positive samples! Adjusting failure horizon...")
    # Try with higher threshold
    for horizon in [50, 75, 100]:
        y_train_bin = (y_train_rul <= horizon).astype(np.float32)
        y_val_bin = (y_val_rul <= horizon).astype(np.float32)
        y_test_bin = (y_test_rul <= horizon).astype(np.float32)
        pos_count = y_train_bin.sum()
        if pos_count > 5:
            neg_count = len(y_train_bin) - pos_count
            pos_weight = torch.tensor([neg_count / max(pos_count, 1)]).to(cfg.DEVICE)
            pred_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            print(f"   Using horizon={horizon}: positive={int(pos_count)} ({pos_count/len(y_train_bin):.1%})")
            break

loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_bin)),
                    batch_size=cfg.PRED_BATCH_SIZE, shuffle=True)

best_f1 = 0
for epoch in range(cfg.PRED_EPOCHS):
    pred.train()
    total_loss = 0
    for bx, by in loader:
        bx, by = bx.to(cfg.DEVICE), by.to(cfg.DEVICE)
        logits, _ = pred(bx)
        loss = pred_criterion(logits, by)
        pred_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(pred.parameters(), 1.0)
        pred_opt.step()
        total_loss += loss.item() * len(bx)

    if (epoch+1) % 10 == 0:
        pred.eval()
        val_proba, _ = pred.predict_proba(torch.FloatTensor(X_val))
        val_pred = (val_proba >= 0.5).astype(int)
        f1 = f1_score(y_val_bin, val_pred, zero_division=0)
        auc = roc_auc_score(y_val_bin, val_proba) if len(np.unique(y_val_bin)) > 1 else 0
        print(f"   Epoch {epoch+1}/{cfg.PRED_EPOCHS} \u2014 Loss: {total_loss/len(X_train):.4f} | F1: {f1:.3f} | AUC: {auc:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            torch.save({"model_state": pred.state_dict(), "input_dim": n_features,
                        "hidden_dim": cfg.PRED_HIDDEN_DIM}, os.path.join(cfg.MODELS_DIR, "ims_predictor.pt"))

if best_f1 == 0 and pos_count > 0:
    # Save current model as fallback even if F1 is 0 on val (may be overfitting val)
    torch.save({"model_state": pred.state_dict(), "input_dim": n_features,
                "hidden_dim": cfg.PRED_HIDDEN_DIM}, os.path.join(cfg.MODELS_DIR, "ims_predictor.pt"))

print(f"\u2705 Best F1: {best_f1:.3f}")


🟡 TRAINING: LSTM FAILURE PREDICTOR (With Attention)
   Samples: 639, pos_rate: 0.00%, weight: 639.00
   Epoch 10/50 — Loss: 0.0034 | F1: 0.000 | AUC: 0.000
   Epoch 20/50 — Loss: 0.0011 | F1: 0.000 | AUC: 0.000
   Epoch 30/50 — Loss: 0.0006 | F1: 0.000 | AUC: 0.000
   Epoch 40/50 — Loss: 0.0004 | F1: 0.000 | AUC: 0.000
   Epoch 50/50 — Loss: 0.0003 | F1: 0.000 | AUC: 0.000
✅ Best F1: 0.000


---
## 9. Model 3: XGBoost RUL Estimation

In [14]:
print("=" * 60)
print("🟢 TRAINING: XGBOOST RUL ESTIMATOR")
print("=" * 60)

exclude_cols = ["file_index", "unit_id", "RUL"]
xgb_feat_cols = [c for c in df_train.columns if c not in exclude_cols]

X_tr_xgb = df_train[xgb_feat_cols]
y_tr_xgb = df_train["RUL"].values
X_va_xgb = df_val[xgb_feat_cols]
y_va_xgb = df_val["RUL"].values
X_te_xgb = df_test[xgb_feat_cols]
y_te_xgb = df_test["RUL"].values

xgb_model = xgb.XGBRegressor(**cfg.XGB_PARAMS)
xgb_model.fit(X_tr_xgb, y_tr_xgb, eval_set=[(X_tr_xgb, y_tr_xgb), (X_va_xgb, y_va_xgb)], verbose=20)

y_pred_xgb = np.clip(xgb_model.predict(X_te_xgb), 0, cfg.IMS_MAX_RUL)
rmse = np.sqrt(mean_squared_error(y_te_xgb, y_pred_xgb))
r2 = r2_score(y_te_xgb, y_pred_xgb)
print(f"\n📊 XGBoost RUL Results:")
print(f"   RMSE: {rmse:.2f} snapshots")
print(f"   R²:   {r2:.4f}")
print(f"   Within ±10: {np.mean(np.abs(y_te_xgb - y_pred_xgb) <= 10)*100:.1f}%")

# Feature importance
imp = pd.DataFrame({"feature": xgb_feat_cols, "importance": xgb_model.feature_importances_})
imp = imp.sort_values("importance", ascending=False).head(15)
print(f"\n   Top 15 features:")
print(imp.to_string(index=False))

joblib.dump({"model": xgb_model, "features": xgb_feat_cols}, os.path.join(cfg.MODELS_DIR, "ims_xgboost.pkl"))
print("✅ XGBoost saved")

🟢 TRAINING: XGBOOST RUL ESTIMATOR
[0]	validation_0-rmse:51.27644	validation_1-rmse:237.82006
[20]	validation_0-rmse:6.87577	validation_1-rmse:118.22092
[40]	validation_0-rmse:1.06209	validation_1-rmse:102.25090
[60]	validation_0-rmse:0.26277	validation_1-rmse:99.90837
[80]	validation_0-rmse:0.12437	validation_1-rmse:99.07635
[100]	validation_0-rmse:0.08502	validation_1-rmse:99.06687
[120]	validation_0-rmse:0.06737	validation_1-rmse:99.03802
[140]	validation_0-rmse:0.05159	validation_1-rmse:99.11326
[160]	validation_0-rmse:0.03923	validation_1-rmse:99.10942
[180]	validation_0-rmse:0.02884	validation_1-rmse:99.11280
[199]	validation_0-rmse:0.02173	validation_1-rmse:99.10986

📊 XGBoost RUL Results:
   RMSE: 239.49 snapshots
   R²:   -30.0037
   Within ±10: 0.0%

   Top 15 features:
                  feature  importance
 bearing1_rms_roll10_mean    0.595835
 bearing1_rms_roll50_mean    0.242070
bearing1_rms_roll100_mean    0.028523
bearing4_rms_roll100_mean    0.027584
 bearing3_rms_roll10

---
## 10. Model 4: Bayesian Weibull Survival Analysis

In [15]:
print("=" * 60)
print("\U0001f535 TRAINING: BAYESIAN WEIBULL SURVIVAL MODEL")
print("=" * 60)

df_surv = df_train[["RUL"] + [c for c in df_train.columns
                               if c not in exclude_cols and df_train[c].dtype in [np.float64, np.float32]]].copy()
df_surv["duration"] = df_surv["RUL"].clip(lower=1)
# Event = bearing has failed or is near failure (RUL <= 30)
df_surv["event"] = (df_surv["RUL"] <= 30).astype(int)
df_surv = df_surv.drop(columns=["RUL"])

event_rate = df_surv["event"].mean()
print(f"   Training event rate: {event_rate:.1%} ({df_surv['event'].sum()}/{len(df_surv)})")

# Limit features to avoid convergence issues
surv_feats = [c for c in df_surv.columns if c not in ["duration", "event"] and df_surv[c].std() > 1e-6]
if len(surv_feats) > 15:
    corrs = df_surv[surv_feats].corrwith(df_surv["duration"]).abs()
    surv_feats = corrs.nlargest(15).index.tolist()
    print(f"   Using top {len(surv_feats)} features by correlation with duration")

df_surv_fit = df_surv[surv_feats + ["duration", "event"]]

try:
    aft = WeibullAFTFitter(penalizer=0.05)
    aft.fit(df_surv_fit, duration_col="duration", event_col="event", show_progress=False)

    # Evaluate on test
    df_test_surv = df_test[surv_feats + ["RUL"]].copy()
    df_test_surv["duration"] = df_test_surv["RUL"].clip(lower=1)
    df_test_surv["event"] = (df_test_surv["RUL"] <= 30).astype(int)
    median_pred = aft.predict_median(df_test_surv[surv_feats]).values.flatten()
    # Replace inf values
    median_pred = np.where(np.isinf(median_pred), cfg.IMS_MAX_RUL * 2, median_pred)
    c_idx = concordance_index(df_test_surv["duration"], median_pred, df_test_surv["event"])
    print(f"\n\U0001f4c8 Survival Model C-Index: {c_idx:.4f}")
    joblib.dump({"model": aft}, os.path.join(cfg.MODELS_DIR, "ims_survival.pkl"))
    print("\u2705 Survival model saved")
except Exception as e:
    print(f"\u26a0\ufe0f Survival model failed: {e}")
    print("   Skipping \u2014 may need more data points for convergence")


🔵 TRAINING: BAYESIAN WEIBULL SURVIVAL MODEL


<lifelines.WeibullAFTFitter: fitted with 688 total observations, 688 right-censored observations>
             duration col = 'duration'
                event col = 'event'
   number of observations = 688
number of events observed = 0
           log-likelihood = -0.000
         time fit was run = 2026-03-10 07:19:28 UTC

---
                                    coef exp(coef)     p
param   covariate                                       
lambda_ bearing1_band_1_energy     1.128     3.090 1.000
        bearing1_band_2_energy     1.541     4.669 1.000
        bearing1_band_3_energy     0.287     1.333 1.000
        bearing1_band_4_energy     1.901     6.693 1.000
        bearing1_band_5_energy     1.319     3.739 1.000
        bearing1_kurtosis          1.251     3.493 1.000
        bearing1_peak_to_peak      1.088     2.969 1.000
        bearing1_rms               0.248     1.282 1.000
        bearing1_rms_roll100_mean  0.355     1.426 1.000
        bearing1_rms_roll100_std   0.337     1.400 1.000
        bearing1_rms_roll10_mean   0.241     1.272 1.000
        bearing1_rms_roll50_mean   0.290     1.336 1.000
        bearing1_rms_roll50_std    0.610     1.841 1.000
        bearing1_std               0.274     1.316 1.000
        bearing4_band_3_energy     0.931     2.537 1.000
        bearing4_rms               1.242     3.464 1.000
        bearing4_rms_roll100_mean  2.816    16.705 1.000
        bearing4_rms_roll10_mean   1.277     3.585 1.000
        bearing4_rms_roll50_mean   1.877     6.536 1.000
        bearing4_std               1.618     5.041 1.000
        Intercept                 11.720 1.231e+05 1.000
rho_    Intercept                  0.542     1.720 1.000
---
Concordance = 0.500
AIC = 44.000
log-likelihood ratio test = -0.000 on 20 df
-log2(p) of ll-ratio test = -0.000


📊 Survival Model C-Index: 0.0516
✅ Survival model saved


---
## 11. Inference Pipeline & Anomaly Detection

In [16]:
print("=" * 60)
print("🔍 INFERENCE PIPELINE")
print("=" * 60)

# Anomaly Detection
if ae.threshold is not None:
    scores, is_anomaly = ae.detect_anomalies(torch.FloatTensor(X_test))
    print(f"\n🔴 Anomaly Detection:")
    print(f"   Detected: {is_anomaly.sum()}/{len(is_anomaly)} ({is_anomaly.mean():.1%})")
    print(f"   Score range: [{scores.min():.6f}, {scores.max():.6f}]")

# Failure Prediction
proba, attn_weights = pred.predict_proba(torch.FloatTensor(X_test))
print(f"\n🟡 Failure Risk:")
print(f"   High risk (>0.7):   {(proba > 0.7).sum()}")
print(f"   Medium (0.4-0.7):   {((proba > 0.4) & (proba <= 0.7)).sum()}")
print(f"   Low risk (<0.4):    {(proba <= 0.4).sum()}")

# RUL Prediction
print(f"\n🟢 RUL Prediction:")
print(f"   RMSE: {rmse:.2f} | R²: {r2:.4f}")

🔍 INFERENCE PIPELINE

🔴 Anomaly Detection:
   Detected: 100/100 (100.0%)
   Score range: [6.302054, 769.662720]

🟡 Failure Risk:
   High risk (>0.7):   0
   Medium (0.4-0.7):   0
   Low risk (<0.4):    100

🟢 RUL Prediction:
   RMSE: 239.49 | R²: -30.0037


---
## 12. MILP Maintenance Scheduling

In [17]:
print("=" * 60)
print("📋 MILP MAINTENANCE SCHEDULING")
print("=" * 60)

# Create bearing risk map
bearing_risks = {}
for b in range(1, exp_info["bearings"] + 1):
    bearing_risks[b] = float(proba[-1])  # Latest prediction

prob = LpProblem("Bearing_Maintenance", LpMinimize)
machines = list(bearing_risks.keys())
n_slots = cfg.SCHEDULING_HORIZON
x_vars = {m: {t: LpVariable(f"x_{m}_{t}", cat=LpBinary) for t in range(n_slots)} for m in machines}

# Objective
obj = []
for m in machines:
    risk = bearing_risks[m]
    for t in range(n_slots):
        obj.append(x_vars[m][t] * cfg.MAINTENANCE_COST_BASE * (1 + 0.1*t))
    not_sched = 1 - lpSum(x_vars[m][t] for t in range(n_slots))
    obj.append(not_sched * risk * cfg.DOWNTIME_COST_PER_HOUR * 8)
prob += lpSum(obj)

# Constraints
for m in machines:
    prob += lpSum(x_vars[m][t] for t in range(n_slots)) <= 1
for t in range(n_slots):
    prob += lpSum(x_vars[m][t] for m in machines) <= cfg.MAX_CONCURRENT_CREWS
for m in machines:
    if bearing_risks[m] >= cfg.SAFETY_RISK_THRESHOLD:
        prob += lpSum(x_vars[m][t] for t in range(n_slots)) >= 1

prob.solve(PULP_CBC_CMD(msg=0, timeLimit=60))
print(f"   Status: {LpStatus[prob.status]}")
print(f"   Total Cost: ${value(prob.objective):,.2f}")

# Print schedule
print(f"\n{'Bearing':<12} {'Risk':>8} {'Level':<22} {'Slot':>6}")
print("-" * 52)
for m in machines:
    risk = bearing_risks[m]
    level = "Service Immediately" if risk >= 0.7 else "Schedule Soon" if risk >= 0.4 else "Monitoring"
    slot = "N/A"
    for t in range(n_slots):
        if value(x_vars[m][t]) == 1:
            slot = str(t); break
    print(f"Bearing-{m:<4} {risk:>8.4f} {level:<22} {slot:>6}")

📋 MILP MAINTENANCE SCHEDULING
   Status: Optimal
   Total Cost: $17.78

Bearing          Risk Level                    Slot
----------------------------------------------------
Bearing-1      0.0001 Monitoring                N/A
Bearing-2      0.0001 Monitoring                N/A
Bearing-3      0.0001 Monitoring                N/A
Bearing-4      0.0001 Monitoring                N/A


---
## 13. Monte Carlo Simulation (Policy Comparison)

In [18]:
print("=" * 60)
print("🎲 MONTE CARLO SIMULATION — Policy Comparison")
print("=" * 60)

rng = np.random.default_rng(42)
n_machines_sim = 50
n_periods_sim = 100
n_sims = 100
failure_cost = cfg.DOWNTIME_COST_PER_HOUR * 16
preventive_cost = cfg.MAINTENANCE_COST_BASE

all_results = []
for sim in range(n_sims):
    # Simulate health
    health = np.ones((n_machines_sim, n_periods_sim))
    for m in range(n_machines_sim):
        shape = rng.uniform(1.5, 3.0)
        scale = rng.uniform(40, 80)
        ft = min(int(rng.weibull(shape) * scale), n_periods_sim)
        for t in range(n_periods_sim):
            health[m, t] = max(0, 1 - (t/ft)**2) if t < ft else 0

    # Reactive
    cost, dt, fails = 0, 0, 0
    h = health.copy()
    for m in range(n_machines_sim):
        for t in range(1, n_periods_sim):
            if h[m,t] == 0 and h[m,t-1] > 0:
                cost += failure_cost; dt += 16; fails += 1
    avail = (1 - dt/(n_machines_sim*n_periods_sim*24))*100
    all_results.append({"sim": sim, "policy": "Reactive", "cost": cost, "downtime": dt,
                        "availability": min(100, avail), "failures": fails, "preventive": 0})

    # Scheduled
    cost, dt, fails, prev = 0, 0, 0, 0
    h = health.copy()
    for m in range(n_machines_sim):
        last = 0
        for t in range(1, n_periods_sim):
            if t - last >= 30:
                cost += preventive_cost; dt += 4; last = t; prev += 1
                h[m,t:] = np.minimum(h[m,t:]+0.5, 1)
            if h[m,t] <= 0.05:
                cost += failure_cost; dt += 16; fails += 1
                h[m,t:] = np.minimum(h[m,t:]+0.8, 1)
    avail = (1 - dt/(n_machines_sim*n_periods_sim*24))*100
    all_results.append({"sim": sim, "policy": "Scheduled", "cost": cost, "downtime": dt,
                        "availability": min(100, avail), "failures": fails, "preventive": prev})

    # Optimized
    cost, dt, fails, prev = 0, 0, 0, 0
    h = health.copy()
    for m in range(n_machines_sim):
        for t in range(1, n_periods_sim):
            hs = h[m,t]
            if hs < 0.4 and hs > 0.05:
                cost += preventive_cost*1.2; dt += 4; prev += 1
                restore = min(0.7, 1-hs)
                h[m,t:] = np.minimum(h[m,t:]+restore, 1)
            elif hs <= 0.05:
                cost += failure_cost; dt += 16; fails += 1
                h[m,t:] = np.minimum(h[m,t:]+0.8, 1)
    avail = (1 - dt/(n_machines_sim*n_periods_sim*24))*100
    all_results.append({"sim": sim, "policy": "Optimized", "cost": cost, "downtime": dt,
                        "availability": min(100, avail), "failures": fails, "preventive": prev})

    if (sim+1) % 25 == 0:
        print(f"   Completed {sim+1}/{n_sims}")

sim_df = pd.DataFrame(all_results)
summary = sim_df.groupby("policy").agg({"cost": ["mean","std"], "downtime": "mean",
                                         "availability": "mean", "failures": "mean"}).round(2)
print(f"\n📊 Results:\n{summary}")

rc = sim_df[sim_df["policy"]=="Reactive"]["cost"].mean()
oc = sim_df[sim_df["policy"]=="Optimized"]["cost"].mean()
print(f"\n💰 Cost reduction (Optimized vs Reactive): {(1-oc/rc)*100:.1f}%")

🎲 MONTE CARLO SIMULATION — Policy Comparison
   Completed 25/100
   Completed 50/100
   Completed 75/100
   Completed 100/100

📊 Results:
                cost            downtime availability failures
                mean        std     mean         mean     mean
policy                                                        
Optimized   181464.0   97370.42   204.68        99.83     0.39
Reactive   7446400.0  311333.14   744.64        99.38    46.54
Scheduled  2008800.0  425987.34   770.88        99.36    10.68

💰 Cost reduction (Optimized vs Reactive): 97.6%


---
## 14. Visualizations

In [19]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Smart Industrial Maintenance — IMS Bearing Analysis", fontsize=16, fontweight='bold')
colors = {"Reactive": "#FF6B6B", "Scheduled": "#FFA726", "Optimized": "#66BB6A"}

# 1. Bearing RMS degradation
ax = axes[0, 0]
for col in rms_cols[:4]:
    ax.plot(df_features["file_index"], df_features[col], alpha=0.7, label=col.replace("_rms",""))
ax.set_xlabel("Snapshot"); ax.set_ylabel("RMS"); ax.set_title("Bearing Vibration RMS Over Time")
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# 2. Cost distribution
ax = axes[0, 1]
for pol in colors:
    ax.hist(sim_df[sim_df["policy"]==pol]["cost"], alpha=0.6, label=pol, color=colors[pol], bins=20)
ax.set_xlabel("Total Cost ($)"); ax.set_title("Cost Distribution (100 MC Sims)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 3. Failure probability over time
ax = axes[0, 2]
ax.plot(proba, color='#FF4444', linewidth=1.5)
ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Critical threshold')
ax.axhline(y=0.4, color='orange', linestyle='--', alpha=0.5, label='Elevated threshold')
ax.set_xlabel("Test sequence"); ax.set_ylabel("P(Failure)")
ax.set_title("LSTM Failure Probability"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 4. XGBoost RUL actual vs predicted
ax = axes[1, 0]
ax.scatter(y_te_xgb, y_pred_xgb, alpha=0.5, s=15, color='#2196F3')
ax.plot([0, max(y_te_xgb)], [0, max(y_te_xgb)], 'r--', label='Perfect')
ax.set_xlabel("Actual RUL"); ax.set_ylabel("Predicted RUL")
ax.set_title(f"XGBoost RUL (R²={r2:.3f})"); ax.legend(); ax.grid(True, alpha=0.3)

# 5. Anomaly scores
if ae.threshold is not None:
    ax = axes[1, 1]
    ax.plot(scores, color='#9C27B0', alpha=0.7, linewidth=1)
    ax.axhline(y=ae.threshold, color='red', linestyle='--', label=f'Threshold={ae.threshold:.4f}')
    ax.fill_between(range(len(scores)), scores, ae.threshold,
                     where=scores > ae.threshold, alpha=0.3, color='red', label='Anomaly')
    ax.set_xlabel("Test sequence"); ax.set_ylabel("Reconstruction Error")
    ax.set_title("Autoencoder Anomaly Detection"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 6. Feature importance (top 10)
ax = axes[1, 2]
top10 = imp.head(10)
ax.barh(range(len(top10)), top10["importance"].values, color='#00BCD4')
ax.set_yticks(range(len(top10)))
ax.set_yticklabels([f.replace("bearing","B").replace("_",".")[:20] for f in top10["feature"].values], fontsize=8)
ax.set_xlabel("Importance"); ax.set_title("Top 10 XGBoost Features"); ax.grid(True, alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(cfg.MODELS_DIR, "full_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plots saved to models/saved/full_analysis.png")

✅ Plots saved to models/saved/full_analysis.png


---
## 15. Summary & Metrics

In [20]:
print("=" * 70)
print("🏭 SMART INDUSTRIAL MAINTENANCE — FINAL RESULTS")
print("=" * 70)

print(f"""
{'Model':<35} {'Metric':<20} {'Value':>10}
{'─'*65}
LSTM Autoencoder                    Threshold            {(f"{ae.threshold:.6f}" if ae.threshold else "N/A")}
                                    Anomalies            {(f"{is_anomaly.sum()}" if ae.threshold else "N/A")}/{len(X_test)}

LSTM Failure Predictor              Best F1              {best_f1:.3f}
                                    High Risk            {(proba > 0.7).sum()}/{len(proba)}

XGBoost RUL                         RMSE                 {rmse:.2f}
                                    R²                   {r2:.4f}

Monte Carlo Simulation              Cost Reduction       {(1-oc/rc)*100:.1f}%
                                    Reactive Cost        ${rc:,.0f}
                                    Optimized Cost       ${oc:,.0f}

Dataset                             Experiment           2 (outer race)
                                    Snapshots            {len(snapshots)}
                                    Features/snapshot    {n_feat}
                                    Sequence length      {cfg.IMS_SEQUENCE_LENGTH}
""")

print("✅ All models trained and evaluated successfully!")
print(f"📁 Models saved to: {cfg.MODELS_DIR}")

🏭 SMART INDUSTRIAL MAINTENANCE — FINAL RESULTS

Model                               Metric                    Value
─────────────────────────────────────────────────────────────────
LSTM Autoencoder                    Threshold            0.034638
                                    Anomalies            100/100

LSTM Failure Predictor              Best F1              0.000
                                    High Risk            0/100

XGBoost RUL                         RMSE                 239.49
                                    R²                   -30.0037

Monte Carlo Simulation              Cost Reduction       97.6%
                                    Reactive Cost        $7,446,400
                                    Optimized Cost       $181,464

Dataset                             Experiment           2 (outer race)
                                    Snapshots            984
                                    Features/snapshot    60
                                    S